# 02 · Action-recognition prompt evaluation (LSO-68)

Scores the **current production prompt** and a set of variants against the same
hand-labelled crops built by `01_build_action_eval_set.ipynb`.

Why this exists: `total_parse_failures` only catches replies that are malformed
or outside the enum. A well-formed answer that is simply *wrong* is invisible
today, so every prompt edit is a guess. This notebook produces the number that
edits get judged against.

A prompt variant is nothing more than a different `ActionConfig` →
a fresh `ActionRecognizer`: `_build_prompt_template()` renders the prompt purely
from `config.actions`, and `_build_response_format()` derives the JSON enum from
the same dict. Nothing in `lum_vision` needs to change to run this sweep.

**Ordering matters** — run `01_...` first, then label the crops, then run this.

In [ ]:
import json
import math
import re
import sys
import time
import warnings
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore")

from lum_vision import ActionConfig, ActionRecognizer

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EVAL = REPO / "notebooks" / "eval" / "action"
CROPS = EVAL / "crops"
CROPS_ASIS = EVAL / "crops_asis"
RESULTS = EVAL / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

OLLAMA_URL = "http://localhost:11534"
MODEL_NAME = "gemma3:4b"

print("eval  :", EVAL)
print("ollama:", OLLAMA_URL, "|", MODEL_NAME)

## 1 · Load the labels

Labels live in the filenames (`phone__cam2_001200_p1_ab12cd34.jpg`) or in a
class subfolder (`crops/phone/...`) — whichever was faster to do by hand. Both
resolve here.

`skip` and still-`unlabeled` crops are excluded from scoring.

In [ ]:
SLUG_TO_ACTION = {
    "sleeping": "sleeping",
    "phone": "using phone",
    "computer": "working with computer",
    "talking": "talking with someone",
    "not_focusing": "not_focusing",
    "idle": "idle",
}
NON_LABELS = {"unlabeled", "skip"}


def resolve_label(path: Path) -> str:
    """Label from the parent folder if it names a class, else the filename prefix."""
    if path.parent.name in SLUG_TO_ACTION or path.parent.name in NON_LABELS:
        return path.parent.name
    return path.stem.split("__", 1)[0]


def load_labels(root: Path) -> tuple[pd.DataFrame, dict]:
    """Walk `root` for labelled crops. Used for both `crops/` and `crops_asis/`."""
    rows, unlabelled, skipped, unknown = [], 0, 0, []
    for p in sorted(root.rglob("*.jpg")):
        slug = resolve_label(p)
        if slug == "unlabeled":
            unlabelled += 1
        elif slug == "skip":
            skipped += 1
        elif slug in SLUG_TO_ACTION:
            rows.append({
                "crop_id": p.stem.split("__", 1)[1] if "__" in p.stem else p.stem,
                "path": str(p),
                "slug": slug,
                "truth": SLUG_TO_ACTION[slug],
            })
        else:
            unknown.append(p.name)
    return pd.DataFrame(rows), {"unlabelled": unlabelled, "skipped": skipped, "unknown": unknown}


labels, _stats = load_labels(CROPS)
print(f"labelled  : {len(labels)}")
print(f"unlabelled: {_stats['unlabelled']}")
print(f"skipped   : {_stats['skipped']}")
if _stats["unknown"]:
    print(f"\n!! {len(_stats['unknown'])} file(s) with an unrecognised prefix - fix or mark 'skip':")
    for name in _stats["unknown"][:10]:
        print("   ", name)

### Class coverage

A class with two examples cannot produce a meaningful accuracy number. The
warning below is the honest signal that a class is too thin to draw conclusions
from — the ticket explicitly prefers reporting the gap over padding the set.

In [ ]:
MIN_PER_CLASS = 5

if labels.empty:
    raise SystemExit(
        "No labelled crops found.\n"
        f"Label the crops in {CROPS} first - see the last cell of 01_build_action_eval_set.ipynb."
    )

coverage = (
    pd.Series({a: 0 for a in SLUG_TO_ACTION.values()})
    .add(labels["truth"].value_counts(), fill_value=0)
    .astype(int)
    .sort_values(ascending=False)
    .rename("n")
    .to_frame()
)
coverage["share"] = (coverage["n"] / len(labels)).map("{:.1%}".format)
display(coverage)

thin = coverage[coverage["n"] < MIN_PER_CLASS]
if not thin.empty:
    print(f"\n!! WARNING - under {MIN_PER_CLASS} examples, per-class scores here are noise:")
    for action, row in thin.iterrows():
        print(f"     {action:<24} n={row['n']}")
    print("\n   Report these as a coverage gap rather than reading their F1 as real.")

# Durable record of the labelling effort: *.jpg and *.csv are both gitignored,
# so JSON is the only form of this that survives a fresh clone.
(EVAL / "labels.json").write_text(
    json.dumps(dict(zip(labels["crop_id"], labels["slug"])), indent=2, sort_keys=True)
)
print(f"\nwrote {(EVAL / 'labels.json').relative_to(REPO)}")

## 2 · Baseline — the prompt production runs today

Built straight from the `action_recognition` block of `configs/config.yaml`, so
this is byte-for-byte the prompt the running service sends.

In [ ]:
cfg_path = REPO / "configs" / "config.yaml"
action_cfg = yaml.safe_load(cfg_path.read_text())["action_recognition"]
BASELINE_ACTIONS = action_cfg["actions"]

# The FIRST LSO-68 round shipped a merge of sleeping/not_focusing into idle, so
# configs/config.yaml now holds a 4-class taxonomy, not the original 6. Several
# variants below (v2, v3, t1-t7) were built as edits *on the original 6*, and
# need all 6 keys to construct - drop_actions(SIX_CLASS_ACTIONS, "sleeping")
# only makes sense if "sleeping" exists to drop. Pinning it as a literal here
# (rather than re-deriving it) keeps those variants reproducible even as
# config.yaml keeps evolving underneath this notebook.
SIX_CLASS_ACTIONS = {
    "sleeping": {"backend_type": "sleeping",
                 "description": "head resting on desk, leaning back with eyes closed, or slumped over"},
    "using phone": {"backend_type": "phone_usage",
                    "description": "holding a phone, looking down at a device in hand"},
    "working with computer": {"backend_type": "working",
                               "description": "sitting at a desk facing a screen or typing"},
    "talking with someone": {"backend_type": "talking",
                              "description": "facing another person, gesturing, or in conversation"},
    "not_focusing": {"backend_type": "not_focusing",
                      "description": "distracted, looking away from work, wandering attention"},
    "idle": {"backend_type": "unknown",
             "description": "standing still, sitting without doing anything specific, looking around"},
}

baseline_config = ActionConfig.from_dict(
    action_cfg, ollama_api_url=OLLAMA_URL, model_name=MODEL_NAME
)
baseline = ActionRecognizer(baseline_config)  # whatever configs/config.yaml ships *today*

print("=" * 72)
print(baseline.prompt_template)
print("=" * 72)
print("\nenum:", baseline.response_format["properties"]["action"]["enum"])

## 3 · Prompt variants

Each variant is a full `actions` dict, so the JSON enum stays consistent with
whatever the prompt describes. Variants that change *instruction wording*
rather than descriptions overwrite `prompt_template` after construction — the
enum is still derived from `actions`, so the decoding contract is untouched.

Scope note: this is prompt text only. Model, crop geometry and the
schema-decoding mechanism are all held fixed, per LSO-68.

In [ ]:
# Defaults copied from _build_prompt_template() so a variant that overrides
# neither renders byte-identically to the library.
DEFAULT_HEADER = "Classify what the person in this image is doing. Pick the best match:\n"
DEFAULT_FOOTER = [
    '\nRespond with JSON: {"action": "<one of the names above>"}',
    'Use "none" only if no option applies.',
]


def build_recognizer(actions: dict, header: str | None = None, footer: list[str] | None = None):
    """An ActionRecognizer over `actions`, optionally with reworded instructions.

    The prompt is always re-rendered here rather than left to the library:
    _build_prompt_template() unconditionally emits "{name} - {description}",
    which leaves a dangling "sleeping - " when a variant drops descriptions
    entirely. The enum still comes from `actions` via _build_response_format(),
    so the decoding contract is untouched.
    """
    cfg = ActionConfig.from_dict(
        {**action_cfg, "actions": actions}, ollama_api_url=OLLAMA_URL, model_name=MODEL_NAME
    )
    rec = ActionRecognizer(cfg)

    lines = [header or DEFAULT_HEADER]
    for i, (name, spec) in enumerate(actions.items(), 1):
        desc = spec.get("description", "")
        lines.append(f"{i}. {name} - {desc}" if desc else f"{i}. {name}")
    lines.extend(footer or DEFAULT_FOOTER)
    rec.prompt_template = "\n".join(lines)
    return rec


def with_descriptions(**overrides) -> dict:
    """Baseline actions with some descriptions swapped, backend_type preserved."""
    out = {name: dict(spec) for name, spec in SIX_CLASS_ACTIONS.items()}
    for name, desc in overrides.items():
        key = name.replace("_", " ") if name.replace("_", " ") in out else name
        out[key]["description"] = desc
    return out


def drop_actions(actions: dict, *names: str) -> dict:
    """Remove actions entirely - from the prompt *and* from the JSON enum.

    This is the whole mechanism behind the taxonomy variants: the enum comes
    from `[*self.actions_config, "none"]` in _build_response_format(), so an
    action absent from the dict simply cannot be returned. No decoding change.
    """
    missing = [n for n in names if n not in actions]
    if missing:  # a silent typo here would look like "the variant didn't help"
        raise KeyError(f"not in action set: {missing} (have: {list(actions)})")
    return {name: dict(spec) for name, spec in actions.items() if name not in names}


def names_only(actions: dict) -> dict:
    """Same actions, descriptions stripped."""
    return {name: {"backend_type": spec["backend_type"]} for name, spec in actions.items()}

In [ ]:
# v1 - concrete visual evidence instead of inferred mental state
V1_ACTIONS = with_descriptions(**{
    "sleeping": "eyes closed, head down on the desk or tipped back, body slack and still",
    "using phone": "a phone or small handheld device is visible in one or both hands",
    "working with computer": "seated facing a monitor, hands on a keyboard, mouse or laptop",
    "talking with someone": "head turned toward another person, mouth open or hands mid-gesture",
    "not_focusing": "seated at a workstation but turned away from the screen, looking elsewhere",
    "idle": "upright and still, no device, screen or other person being engaged with",
})

# v2 - an explicit precedence rule for the overlap that should bite most:
# someone at a desk holding a phone satisfies two descriptions at once.
V2_FOOTER = [
    "\nTie-breaks, in order:",
    "- A phone visible in hand means 'using phone', even if a monitor is also in view.",
    "- Hands on a keyboard or mouse means 'working with computer'.",
    "- Facing another person means 'talking with someone'.",
    '\nRespond with JSON: {"action": "<one of the names above>"}',
    'Use "none" only if no option applies.',
]

# v3 - names only. Tests whether the descriptions earn their place at all.
V3_ACTIONS = {name: {"backend_type": spec["backend_type"]} for name, spec in SIX_CLASS_ACTIONS.items()}

# v4 - single-still framing, aimed at the not_focusing/idle ambiguity the ticket
# names: 'not_focusing' is a claim about attention over time, unknowable from one frame.
V4_HEADER = (
    "You are looking at a single still crop from a security camera. "
    "Judge only what is visible - do not infer movement, intent or what happened "
    "before or after.\nClassify what the person is doing. Pick the best match:\n"
)
V4_FOOTER = [
    "\nIf no specific activity is clearly visible, answer 'idle' rather than guessing.",
    '\nRespond with JSON: {"action": "<one of the names above>"}',
    'Use "none" only if no option applies.',
]

VARIANTS = {
    "baseline":           baseline,   # whatever configs/config.yaml ships today
    # The original 6-class prompt, before round 1 shipped the sleeping/
    # not_focusing -> idle merge. Kept so the very first number this notebook
    # ever produced (29.0% accuracy) stays directly reproducible, now scored
    # against a much bigger and better-covered crop set than it had then.
    "pre_merge_6class":   build_recognizer(SIX_CLASS_ACTIONS),
    "v1_visual_evidence": build_recognizer(V1_ACTIONS),
    "v2_tiebreak":        build_recognizer(SIX_CLASS_ACTIONS, footer=V2_FOOTER),
    "v3_names_only":      build_recognizer(V3_ACTIONS),
    "v4_single_frame":    build_recognizer(V1_ACTIONS, header=V4_HEADER, footer=V4_FOOTER),
}

### Taxonomy variants

The round-one results pointed at one dominant cause: **`not_focusing` has zero
ground-truth labels in this set, yet the baseline predicts it 15 of 31 times** -
48% of all predictions, every one of them wrong by construction. Its description
("distracted, looking away from work, wandering attention") names a mental state
*over time*, which a single still frame cannot show, so the model falls back to
it whenever it is unsure. `sleeping` behaves the same way on a smaller scale.

Remapping just those two predictions to `idle` moves the baseline 29.0% -> 64.5%
and `v3_names_only` 38.7% -> 77.4%. These variants test that directly, by
removing or tightening the categories rather than assuming the remap.

Dropping an action removes it from the prompt **and** the JSON enum, since
`_build_response_format()` builds the enum from the same dict.

In [ ]:
# t3 - keep all six, but gate the two catch-alls behind evidence you can
# actually see in one frame, instead of a state you would have to infer.
STRICT_ACTIONS = with_descriptions(**{
    "not_focusing": "a screen is visible in frame AND the person's head is clearly turned away from it",
    "sleeping": "eyes visibly closed AND head resting on a desk or tipped fully back",
})

# t6 - t3 plus a soft nudge for the uncertain case. Watch this one: v4 showed
# that a hard "answer idle when unsure" collapses the model onto one class.
T6_FOOTER = [
    "\nPrefer 'idle' over 'not_focusing' when you cannot see what has their attention.",
    '\nRespond with JSON: {"action": "<one of the names above>"}',
    'Use "none" only if no option applies.',
]

# t7 - same gates, commonest classes first, to test list-position bias.
T7_ACTIONS = {
    name: STRICT_ACTIONS[name]
    for name in ["idle", "working with computer", "using phone",
                 "talking with someone", "not_focusing", "sleeping"]
}

VARIANTS.update({
    "t1_drop_notfocus":           build_recognizer(drop_actions(SIX_CLASS_ACTIONS, "not_focusing")),
    "t2_drop_notfocus_sleeping":  build_recognizer(drop_actions(SIX_CLASS_ACTIONS, "not_focusing", "sleeping")),
    "t3_strict_gates":            build_recognizer(STRICT_ACTIONS),
    "t4_names_only_drop_notfocus": build_recognizer(names_only(drop_actions(SIX_CLASS_ACTIONS, "not_focusing"))),
    "t5_names_only_4class":       build_recognizer(names_only(drop_actions(SIX_CLASS_ACTIONS, "not_focusing", "sleeping"))),
    "t6_strict_plus_default":     build_recognizer(STRICT_ACTIONS, footer=T6_FOOTER),
    "t7_reorder":                 build_recognizer(T7_ACTIONS),
})

### Merge variants — one bucket for "not doing anything in particular"

Dropping `not_focusing` and `sleeping` leaves a genuinely sleeping person with
nowhere correct to go; the model has to pick a wrong answer. **Merging** the
three low-signal states — `idle`, `not_focusing`, `sleeping` — into a single
bucket keeps a correct destination for all of them while removing the
distinction the model cannot make from one still frame.

That is also the distinction the ticket already flagged as a known limitation,
so collapsing it is honest rather than a dodge.

The merged bucket keeps `backend_type: unknown` (what `idle` already maps to),
so nothing new is ever written to `activity_type` and the Postgres CHECK
constraint in `so.stack` is untouched.

`m3` renames the bucket to `inactive` with an identical description, to test
whether the word "idle" itself is steering the model.

In [ ]:
def merged_actions(bucket_name="idle", description=None, keep_descriptions=True):
    """The 4-class taxonomy: 3 activities + one merged low-signal bucket."""
    out = {}
    for name in ["using phone", "working with computer", "talking with someone"]:
        spec = dict(SIX_CLASS_ACTIONS[name])
        if not keep_descriptions:
            spec.pop("description", None)
        out[name] = spec
    bucket = {"backend_type": "unknown"}
    if keep_descriptions:
        bucket["description"] = description or (
            "none of the above - sitting or standing still, resting, looking away, "
            "eyes closed, or asleep"
        )
    out[bucket_name] = bucket
    return out


VARIANTS.update({
    "m1_merge3":            build_recognizer(merged_actions()),
    "m2_merge3_names_only": build_recognizer(merged_actions(keep_descriptions=False)),
    "m3_merge3_inactive":   build_recognizer(merged_actions(bucket_name="inactive")),
})

for name, rec in VARIANTS.items():
    print(f"\n{'=' * 72}\n### {name}   (enum: {rec.response_format['properties']['action']['enum']})\n{'=' * 72}")
    print(rec.prompt_template)

## 4 · Run the sweep

Results stream to `results/<variant>.jsonl` as they arrive, so a crash mid-sweep
loses nothing and a re-run resumes from where it stopped.

Two things deliberately **do not** get served from cache:

- **Edit a variant's wording** and its cached rows are discarded — they carry a
  hash of the prompt that produced them. Iterating on a prompt is the point of
  this notebook; a cache that outlived the edit would report the old prompt's
  score under the new prompt's name.
- **Relabel a crop** and scoring picks up the new label without re-running
  inference — a prediction depends on the image and the prompt, not on what you
  called it.

`temperature=0` is already set inside `recognize_via_api`, so repeat runs are
deterministic.

In [ ]:
import hashlib


def prompt_hash(recognizer) -> str:
    return hashlib.sha1(recognizer.prompt_template.encode()).hexdigest()[:12]


def load_done(path: Path, expected_hash: str) -> dict:
    """Cached rows for this variant, ignoring any produced by an older prompt.

    Editing a variant's wording and re-running is the whole point of this
    notebook, so a cache that survived the edit would quietly report the old
    prompt's score under the new prompt's name.
    """
    if not path.exists():
        return {}
    done, stale = {}, 0
    for line in path.read_text().splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)
        if rec.get("prompt_hash") != expected_hash:
            stale += 1
            continue
        done[rec["crop_id"]] = rec
    if stale:
        print(f"{'':<20} ({stale} row(s) from an older prompt ignored)")
    return done


def run_variant(name: str, recognizer, df: pd.DataFrame, results_dir: Path = RESULTS) -> pd.DataFrame:
    """Score one prompt over every labelled crop, resuming if partly done."""
    out_path = results_dir / f"{name}.jsonl"
    phash = prompt_hash(recognizer)
    done = load_done(out_path, phash)
    todo = df[~df["crop_id"].isin(done)]
    print(f"{name:<20} {len(done)} cached, {len(todo)} to run")

    if len(todo):
        # Ollama unloads an idle model after ~5 min and a cold gemma3:4b load
        # costs ~28s. Absorb that once, outside the measured calls.
        recognizer.recognize(cv2.imread(todo.iloc[0]["path"]))

        with out_path.open("a") as fh:
            for n, row in enumerate(todo.itertuples(), 1):
                before = recognizer.get_stats()
                t0 = time.perf_counter()
                result = recognizer.recognize(cv2.imread(row.path), metadata={"crop_id": row.crop_id})
                elapsed = time.perf_counter() - t0
                after = recognizer.get_stats()

                rec = {
                    "crop_id": row.crop_id,
                    "prompt_hash": phash,
                    "truth": row.truth,
                    "pred": result.action if result else None,
                    "raw": result.raw_output if result else "",
                    "latency_s": round(elapsed, 3),
                    "failed": result is None,
                    "parse_failure": after["total_parse_failures"] > before["total_parse_failures"],
                    "timeout": after["total_timeouts"] > before["total_timeouts"],
                }
                fh.write(json.dumps(rec) + "\n")
                fh.flush()
                done[row.crop_id] = rec
                if n % 20 == 0:
                    print(f"  {n}/{len(todo)}")

    # Take the *current* label, never the one cached alongside the prediction.
    # A prediction depends only on the image and the prompt, so relabelling a
    # crop must not require re-running inference - but it must not silently
    # score against the stale label either.
    res = pd.DataFrame(done.values()).drop(columns=["truth"])
    return res.merge(df[["crop_id", "truth"]], on="crop_id", how="inner")


runs = {name: run_variant(name, rec, labels) for name, rec in VARIANTS.items()}


def offered_actions(recognizer) -> set:
    return set(recognizer.response_format["properties"]["action"]["enum"]) - {"none"}


PRED_ALIASES = {"inactive": "idle"}  # m3 names its bucket "inactive", not "idle"


def normalize_scoring(runs: dict) -> None:
    """Fold each variant's own answer space back into the shared truth space.

    Any variant that dropped or merged an action (t1/t2/t5, m1-m3) cannot
    possibly answer "sleeping" or "not_focusing" - it was never offered the
    option. Scoring those crops against the original 6-class truth would mark
    every one of them wrong regardless of what the model said, which measures
    the taxonomy change, not the model. So: a ground-truth label absent from a
    variant's own enum is remapped to "idle" for that variant only, before
    accuracy is computed - "idle" is present in every variant here and is what
    the dropped/merged classes represent in every case.
    Mutates each DataFrame in place; the underlying .jsonl is untouched, so
    this can be re-derived any time from raw model output.
    """
    for name, res in runs.items():
        res["pred"] = res["pred"].replace(PRED_ALIASES)
        offered = offered_actions(VARIANTS[name])
        remap = {label: "idle" for label in SLUG_TO_ACTION.values() if label not in offered and label != "idle"}
        if remap:
            res["truth"] = res["truth"].replace(remap)


normalize_scoring(runs)
print("\ndone")

## 5 · Score

`pred` is `None` when the model answered "none", replied outside the enum, or
the call failed outright. Those count as **wrong**, not as missing data — a
production frame with no action is a frame the feature did nothing useful on.

In [ ]:
from sklearn.metrics import balanced_accuracy_score

ACTIONS = list(SIX_CLASS_ACTIONS)


def score(name: str, res: pd.DataFrame) -> dict:
    correct = (res["pred"] == res["truth"]).sum()
    preds = res["pred"].fillna("<no answer>")
    return {
        "variant": name,
        "n": len(res),
        "accuracy": correct / len(res) if len(res) else 0.0,
        # Mean of per-class recall - unlike raw accuracy this can't be gamed by
        # always guessing the majority label. With idle at 48% of this eval set,
        # a variant that answers "idle" every time scores ~48% raw accuracy
        # while contributing nothing - balanced accuracy catches that.
        "balanced_accuracy": balanced_accuracy_score(res["truth"], preds),
        "predicted_classes": preds.nunique(),
        "mean_latency_s": res["latency_s"].mean(),
        "p95_latency_s": res["latency_s"].quantile(0.95),
        "parse_failures": int(res["parse_failure"].sum()),
        "timeouts": int(res["timeout"].sum()),
        "no_answer": int(res["pred"].isna().sum()),
    }


summary = pd.DataFrame([score(n, r) for n, r in runs.items()]).set_index("variant")
base = summary.loc["baseline"]
summary["acc_delta"] = summary["accuracy"] - base["accuracy"]
summary["bal_acc_delta"] = summary["balanced_accuracy"] - base["balanced_accuracy"]

display(
    summary.sort_values("accuracy", ascending=False).style.format({
        "accuracy": "{:.1%}", "acc_delta": "{:+.1%}",
        "balanced_accuracy": "{:.1%}", "bal_acc_delta": "{:+.1%}",
        "mean_latency_s": "{:.2f}", "p95_latency_s": "{:.2f}",
    })
)

### How much of this is real? — bootstrap stability

With 31 crops and no held-out split, two things inflate a winner: **one crop is
worth 3.2%**, and picking the best of a dozen variants scored on the same crops
rewards luck as readily as skill.

The bootstrap below resamples the eval set 2000x and reports, per variant, a 95%
confidence interval on accuracy and **P(beats baseline)** — the fraction of
resamples where it still wins. A genuine improvement wins in most resamples; a
lucky one does not. Treat anything under ~80% here as not yet demonstrated.

In [ ]:
RNG_SEED = 12345
N_BOOT = 2000

crop_ids = list(runs["baseline"]["crop_id"])
correct = {}  # variant -> bool array aligned on crop_ids
for name, res in runs.items():
    aligned = res.set_index("crop_id").reindex(crop_ids)
    correct[name] = (aligned["pred"] == aligned["truth"]).to_numpy()

rng = np.random.default_rng(RNG_SEED)
idx = rng.integers(0, len(crop_ids), size=(N_BOOT, len(crop_ids)))

boot = {name: correct[name][idx].mean(axis=1) for name in runs}
base_boot = boot["baseline"]

stability = pd.DataFrame({
    "accuracy":      {n: correct[n].mean() for n in runs},
    "ci_low":        {n: np.percentile(boot[n], 2.5) for n in runs},
    "ci_high":       {n: np.percentile(boot[n], 97.5) for n in runs},
    "p_beats_base":  {n: (boot[n] > base_boot).mean() for n in runs},
    "p_over_80pct":  {n: (boot[n] >= 0.80).mean() for n in runs},
}).sort_values("accuracy", ascending=False)

display(stability.style.format({
    "accuracy": "{:.1%}", "ci_low": "{:.1%}", "ci_high": "{:.1%}",
    "p_beats_base": "{:.0%}", "p_over_80pct": "{:.0%}",
}))

print(f"\nn = {len(crop_ids)} crops - one crop is worth {1/len(crop_ids):.1%}, "
      f"and {math.ceil(0.80 * len(crop_ids))} of {len(crop_ids)} correct is needed for 80%.")

### Regression gates

A variant only ships if it beats the baseline on **both** raw and balanced
accuracy, without costing latency or well-formedness.

Raw accuracy alone is gameable whenever the eval set is class-imbalanced (it
is here — `idle` is roughly half the labelled crops): a variant that just
always answers "idle" scores ~48% raw accuracy while distinguishing nothing.
Balanced accuracy (mean of per-class recall) is what actually catches that,
so it is a gate here, not just a reported number.

In [ ]:
LATENCY_TOLERANCE = 1.10  # allow 10% slower before calling it a regression

print(f"baseline: {base['accuracy']:.1%} accuracy, {base['balanced_accuracy']:.1%} balanced, "
      f"{base['mean_latency_s']:.2f}s mean, {int(base['parse_failures'])} parse failures\n")

for name, row in summary.drop(index="baseline").iterrows():
    checks = {
        "accuracy":          (row["accuracy"] > base["accuracy"],
                              f"{row['accuracy']:.1%} vs {base['accuracy']:.1%}"),
        "balanced accuracy": (row["balanced_accuracy"] > base["balanced_accuracy"],
                              f"{row['balanced_accuracy']:.1%} vs {base['balanced_accuracy']:.1%}"),
        "latency":           (row["mean_latency_s"] <= base["mean_latency_s"] * LATENCY_TOLERANCE,
                              f"{row['mean_latency_s']:.2f}s vs {base['mean_latency_s']:.2f}s"),
        "parse failures":    (row["parse_failures"] <= base["parse_failures"],
                              f"{int(row['parse_failures'])} vs {int(base['parse_failures'])}"),
    }
    verdict = "SHIPPABLE" if all(ok for ok, _ in checks.values()) else "no"
    print(f"{name:<20} {verdict}")
    for label, (ok, detail) in checks.items():
        print(f"    {'PASS' if ok else 'FAIL'}  {label:<18} {detail}")
    print()

### Per-class breakdown and confusion

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

best = summary["balanced_accuracy"].idxmax()
print(f"best variant: {best}\n")

for name in ("baseline", best) if best != "baseline" else ("baseline",):
    res = runs[name]
    print(f"{'=' * 60}\n{name}\n{'=' * 60}")
    print(classification_report(
        res["truth"], res["pred"].fillna("<no answer>"),
        labels=ACTIONS, zero_division=0,
    ))

In [ ]:
def plot_confusion(name: str, ax):
    res = runs[name]
    rows = ACTIONS
    cols = ACTIONS + ["<no answer>"]
    cm = confusion_matrix(res["truth"], res["pred"].fillna("<no answer>"), labels=cols)
    cm = cm[: len(rows), :]

    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(cols)), cols, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(rows)), rows, fontsize=8)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(f"{name} - {summary.loc[name, 'accuracy']:.1%}")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if cm[i, j]:
                ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                        color="white" if cm[i, j] > cm.max() / 2 else "black")


names = ["baseline"] + [n for n in runs if n != "baseline"]
fig, axes = plt.subplots(1, len(names), figsize=(6 * len(names), 5))
for ax, name in zip(np.atleast_1d(axes), names):
    plot_confusion(name, ax)
plt.tight_layout()
plt.savefig(EVAL / "confusion_matrices.png", dpi=110, bbox_inches="tight")
plt.show()

### Most-confused pairs

Pairs that stay confused across *every* variant are the ones that are genuinely
undecidable from a single static crop. LSO-68 asks for those to be written down
as a known limitation rather than iterated on forever.

In [ ]:
pairs = Counter()
per_variant = {}
for name, res in runs.items():
    local = Counter()
    for truth, pred in zip(res["truth"], res["pred"].fillna("<no answer>")):
        if truth != pred:
            local[(truth, pred)] += 1
    per_variant[name] = local
    pairs.update(local)

print(f"{'true':<24} {'predicted':<24} {'total':>6}  {'in all variants':>16}")
print("-" * 76)
for (truth, pred), n in pairs.most_common(12):
    everywhere = all((truth, pred) in per_variant[v] for v in runs)
    print(f"{truth:<24} {pred:<24} {n:>6}  {'yes' if everywhere else '':>16}")

### Look at what the best variant got wrong

Misclassifications are worth eyeballing before trusting the number — some will
be genuine model errors, and some will be crops that were mislabelled or should
have been marked `skip`.

In [ ]:
def show_errors(name: str, limit: int = 12):
    res = runs[name].merge(labels[["crop_id", "path"]], on="crop_id")
    errors = res[res["pred"] != res["truth"]].head(limit)
    if errors.empty:
        print("no errors")
        return

    cols = 6
    rows = int(np.ceil(len(errors) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 5 * rows))
    for ax, (_, row) in zip(np.atleast_1d(axes).ravel(), errors.iterrows()):
        img = cv2.imread(row["path"])
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"true: {row['truth']}\npred: {row['pred']}", fontsize=8)
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    plt.suptitle(f"{name} - misclassified", y=1.0)
    plt.tight_layout()
    plt.show()


show_errors(best)

## 6 · Outcome

The cell below writes the run to `eval/action/summary.json` — the record LSO-68
asks for. Read the regression gates above before changing anything:

- If a variant is **SHIPPABLE**, port its wording into
  `action_recognition.actions` in `configs/config.yaml`. Variants that changed
  the instruction *header/footer* rather than the descriptions need a
  `_build_prompt_template()` change in the `lum-model-vision` repo — a separate
  PR, not a config edit.
- If nothing beat the baseline, that is a valid documented outcome. Record the
  baseline number and leave the prompt alone.

In [ ]:
outcome = {
    "model": MODEL_NAME,
    "eval_set_size": len(labels),
    "class_coverage": labels["truth"].value_counts().to_dict(),
    "thin_classes": thin.index.tolist(),
    "variants": json.loads(summary.reset_index().to_json(orient="records")),
    "best_variant": best,
    "beat_baseline": bool(summary.loc[best, "balanced_accuracy"] > base["balanced_accuracy"]),
    "persistent_confusions": [
        {"true": t, "pred": p, "count": n}
        for (t, p), n in pairs.most_common(12)
        if all((t, p) in per_variant[v] for v in runs)
    ],
}
(EVAL / "summary.json").write_text(json.dumps(outcome, indent=2))

print(f"baseline: {base['accuracy']:.1%} accuracy, {base['balanced_accuracy']:.1%} balanced  ({len(labels)} crops)")
print(f"best (by balanced accuracy): {best} @ {summary.loc[best, 'accuracy']:.1%} accuracy, "
      f"{summary.loc[best, 'balanced_accuracy']:.1%} balanced")
if outcome["beat_baseline"]:
    print(f"\n=> {best} beats the baseline by {summary.loc[best, 'bal_acc_delta']:+.1%} balanced accuracy.")
    print("   Check its regression gates above before shipping it.")
else:
    print("\n=> No variant beat the baseline on balanced accuracy. Leave the prompt as is;")
    print("   record this baseline as the number future attempts must clear.")
    high_raw = summary[(summary["accuracy"] > base["accuracy"]) & (summary.index != "baseline")]
    if not high_raw.empty:
        print(f"\n   Note: {', '.join(high_raw.index)} scored higher on raw accuracy alone -")
        print("   check predicted_classes above; a variant that collapses to one class")
        print("   can out-score the baseline on accuracy without actually being better.")

print(f"\nwrote {(EVAL / 'summary.json').relative_to(REPO)}")

## 7 · As-is ceiling — does crop geometry explain the "talking" failure?

Every `talking with someone` crop in the production-faithful set above was
re-detected and tight-cropped to one person — the QA sheet in notebook 1
showed the conversation partner cut out of frame in every single case. This
section runs the identical variants against `crops_asis/`, where the
smart-office images are used exactly as given, both people still in frame, to
test whether the prompt would work at all if crop geometry were ever
revisited. Changing production's crop is out of scope for this ticket — this
is only evidence for whether it would be worth doing.

In [ ]:
labels_asis, stats_asis = load_labels(CROPS_ASIS)
print(f"crops_asis labelled: {len(labels_asis)}")
if stats_asis["unknown"]:
    print("unrecognised:", stats_asis["unknown"])

display(labels_asis["truth"].value_counts().rename("n").to_frame())

In [ ]:
RESULTS_ASIS = EVAL / "results_asis"
RESULTS_ASIS.mkdir(parents=True, exist_ok=True)

runs_asis = {name: run_variant(name, rec, labels_asis, results_dir=RESULTS_ASIS) for name, rec in VARIANTS.items()}
normalize_scoring(runs_asis)
print("\ndone")

In [ ]:
summary_asis = pd.DataFrame([score(n, r) for n, r in runs_asis.items()]).set_index("variant")
base_asis = summary_asis.loc["baseline"]
summary_asis["acc_delta"] = summary_asis["accuracy"] - base_asis["accuracy"]
summary_asis["bal_acc_delta"] = summary_asis["balanced_accuracy"] - base_asis["balanced_accuracy"]

display(
    summary_asis.sort_values("accuracy", ascending=False).style.format({
        "accuracy": "{:.1%}", "acc_delta": "{:+.1%}",
        "balanced_accuracy": "{:.1%}", "bal_acc_delta": "{:+.1%}",
        "mean_latency_s": "{:.2f}", "p95_latency_s": "{:.2f}",
    })
)

### Production crop vs as-is, per variant

Same variant, same underlying image, only the framing differs. A large gap
here is crop geometry costing accuracy; a small gap says the prompt — not the
crop — is the bottleneck.

In [ ]:
compare = pd.DataFrame({
    "production_acc": summary["accuracy"],
    "asis_acc": summary_asis["accuracy"],
}).dropna()
compare["gap"] = compare["asis_acc"] - compare["production_acc"]
display(compare.sort_values("gap", ascending=False).style.format("{:.1%}"))


def talking_recall(runs_dict, name):
    res = runs_dict[name]
    sub = res[res["truth"] == "talking with someone"]
    return (sub["pred"] == sub["truth"]).mean() if len(sub) else float("nan")


print("\n\'talking with someone\' recall, production crop vs as-is:")
for name in VARIANTS:
    p_r, a_r = talking_recall(runs, name), talking_recall(runs_asis, name)
    p_str = f"{p_r:5.1%}" if p_r == p_r else "  n/a"
    a_str = f"{a_r:5.1%}" if a_r == a_r else "  n/a"
    print(f"  {name:30} production={p_str}  as-is={a_str}")